In [35]:
import pandas as pd
import numpy as np
from unidecode import unidecode

In [36]:

# Danh sách các phần tiền tố (không bao gồm phần 'Bundesliga')
file_parts = [
    "base_info",
    "data_details_Aerial",
    "data_details_Assists",
    "data_details_Blocks",
    "data_details_Cards",
    "data_details_Clearances",
    "data_details_Dribbles",
    "data_details_Fouls",
    "data_details_Goals",
    "data_details_Interception",
    "data_details_Key passes",
    "data_details_Offsides",
    "data_details_Passes",
    "data_details_Possession loss",
    "data_details_Saves",
    "data_details_Shots",
    "data_details_Tackles",
    "data_summary"
]

# Tạo danh sách file dành cho EPL
filenames = [f"{part}_EPL.csv" for part in file_parts]

# Đọc các file CSV
df_list = [pd.read_csv(file) for file in filenames]

# Ghép lại theo chiều ngang
data = pd.concat(df_list, axis=1)

# Hiển thị kết quả
data.head()


,Name,Club,Age,Height,National,Positions,Unnamed: 0,TotAD,WonAD,LostAD,...,Mins,Goals,Assists,Yel,Red,SpG,PS%,AerialsWon,MotM,Rating
0,Mohamed Salah,Liverpool,32 years old (15-06-1992),175cm,Egypt,"Attacking Midfielder (Centre, Left, Right), Fo...",0,0.7,0.3,0.4,...,2840,27,18,1,-,3.4,74.3,0.3,10,7.75
1,Bukayo Saka,Arsenal,23 years old (05-09-2001),178cm,England,"Defender (Left), Midfielder (Centre, Left, Right)",1,1.4,0.4,0.9,...,1372,6,10,3,-,2.7,84.3,0.4,6,7.59
2,Alexander Isak,Newcastle,25 years old (21-09-1999),192cm,Sweden,"Attacking Midfielder (Left), Forward",2,2.5,0.9,1.6,...,2351,21,6,1,-,2.9,75.2,0.9,5,7.44
3,Matheus Cunha,Wolves,25 years old (27-05-1999),183cm,Brazil,"Attacking Midfielder (Centre, Left), Forward",3,1.2,0.4,0.9,...,2168,14,4,3,-,3.3,77.9,0.4,8,7.37
4,Erling Haaland,Manchester City,24 years old (21-07-2000),194cm,Norway,Forward,4,3.4,1.8,1.6,...,2484,21,3,2,-,3.6,66.4,1.8,6,7.37


In [37]:
data= data.drop(columns= ['Unnamed: 0', 'Name'])
data['League'] = 'EPL'

In [38]:
count_club = data['Club'].value_counts()
count_club

Club
Leicester               30
Ipswich                 30
West Ham                29
Tottenham               29
Manchester City         28
Southampton             28
Chelsea                 27
Fulham                  27
Brighton                26
Wolves                  26
Manchester United       26
Bournemouth             25
Aston Villa             25
Everton                 25
Arsenal                 24
Liverpool               24
Brentford               24
Crystal Palace          23
Nottingham Forest       22
Newcastle               22
West Bromwich Albion     3
Napoli                   3
Sheffield United         3
Derby                    2
AC Milan                 2
Middlesbrough            2
Juventus                 2
Aberdeen                 1
Bayer Leverkusen         1
Al Shabab                1
Celtic                   1
Strasbourg               1
Queens Park Rangers      1
PSV Eindhoven            1
Preston                  1
Birmingham               1
Cardiff                

In [39]:
data = data[data['Club'].isin(count_club[count_club > 20].index)]

In [40]:
player_count = data['Player'].value_counts()
player_count

Player
Julio Enciso       2
Jaden Philogene    2
Axel Disasi        2
Carlos Alcaraz     2
Jordan Ayew        2
                  ..
Chris Richards     1
Antonín Kinsky     1
Joël Veltman       1
Jan Bednarek       1
Altay Bayindir     1
Name: count, Length: 508, dtype: int64

In [41]:
invalid_player = data[data.duplicated(subset='Player', keep=False)]


for p in invalid_player['Player'].unique():
    
    players = data[data['Player'] == p]
    
    # Tìm phần tử có số phút thi đấu Mins lớn nhất
    player_to_keep = players.loc[players['Mins'].idxmax()]
    
    # Xóa các phần tử còn lại
    data = data.drop(players[players.index != player_to_keep.name].index)




In [42]:
data_info = data.sort_values(by = 'Player', ascending= True)
data_value = pd.read_csv('D:\Python_Code\Project_DS108\ds108\Transfermark \EPL\EPL_Value.csv')


In [43]:
#data_value['Name'] = data_value['Name'].apply(unidecode)
#data_info['Player'] = data_info['Player'].apply(unidecode)
# Bỏ dấu, chuyển chữ thường và loại dấu gạch nối
data_value['Name'] = data_value['Name'].apply(lambda x: unidecode(x).lower().replace("-", " "))
# Bỏ dấu, chuyển chữ thường và thay dấu gạch nối bằng dấu cách
data_info['Player'] = data_info['Player'].apply(lambda x: unidecode(x).lower().replace("-", " "))



In [44]:
merged_rows = []
name_not_found = []
not_found_indices = []

for i in range(len(data_info)):
    flag = 0
    for j in range(len(data_value)):
        if data_info.iloc[i]['Player'] == data_value.iloc[j]['Name']:
            # Nối 2 hàng lại theo chiều ngang
            merged = pd.concat([
                data_info.iloc[i],
                data_value.iloc[j].drop('Name')  # bỏ cột trùng
            ])
            merged_rows.append(merged)

            flag = 1
            break
    if flag == 0:
        print('Index = ', i, 'Not Found : ', data_info.iloc[i]['Player'])
        name_not_found.append(data_info.iloc[i]['Player'])
        not_found_indices.append(i)




Index =  5 Not Found :  abdul fatawu
Index =  19 Not Found :  alfie pond
Index =  20 Not Found :  alisson becker
Index =  29 Not Found :  andy irving
Index =  30 Not Found :  andy robertson
Index =  51 Not Found :  ben winterburn
Index =  85 Not Found :  chido obi
Index =  125 Not Found :  divin mubama
Index =  170 Not Found :  harry amass
Index =  177 Not Found :  hwang hee chan
Index =  178 Not Found :  hakon valdimarsson
Index =  183 Not Found :  igor julio
Index =  187 Not Found :  illia zabarnyi
Index =  201 Not Found :  jahmai simpson pusey
Index =  202 Not Found :  jake evans
Index =  223 Not Found :  jay robinson
Index =  224 Not Found :  jayden danns
Index =  230 Not Found :  jeremy monga
Index =  271 Not Found :  kepa
Index =  278 Not Found :  kim ji soo
Index =  281 Not Found :  kostas tsimikas
Index =  295 Not Found :  lewis orford
Index =  358 Not Found :  mykhailo mudryk
Index =  383 Not Found :  ollie scarles
Index =  390 Not Found :  pape sarr
Index =  409 Not Found :  

In [45]:
for idx, name in zip(not_found_indices, name_not_found):
    last_name_1 = name.split(' ')[-1]
    found = False
    for j in range(len(data_value)):
        last_name_2 = data_value.iloc[j]['Name'].split(' ')[-1]
        if last_name_1 == last_name_2:
            merged = pd.concat([
                data_info.iloc[idx],
                data_value.iloc[j].drop('Name')  # bỏ cột trùng
            ])
            merged_rows.append(merged)
            found = True
            break
    if not found:
        print('Still not found:', name)


Still not found: alfie pond
Still not found: alisson becker
Still not found: ben winterburn
Still not found: chido obi
Still not found: divin mubama
Still not found: harry amass
Still not found: hwang hee chan
Still not found: igor julio
Still not found: jahmai simpson pusey
Still not found: jayden danns
Still not found: jeremy monga
Still not found: kepa
Still not found: kim ji soo
Still not found: lewis orford
Still not found: remy rees dottin
Still not found: roman dixon
Still not found: shumaira mheuka
Still not found: son heung min
Still not found: yehor yarmoliuk
Still not found: zain silcott duberry


In [46]:
for i in range(len(merged_rows)):
    merged_rows[i] = merged_rows[i].to_dict()
data = pd.DataFrame(merged_rows)

data['Player'] = data['Player'].str.title()
data.to_csv('Final_EPL.csv')

In [47]:
data['Player']

0         Aaron Cresswell
1          Aaron Ramsdale
2       Aaron Wan Bissaka
3      Abdoulaye Doucoure
4      Abdukodir Khusanov
              ...        
483            Toti Gomes
484     Vitalii Mykolenko
485     Willy Arnaud Boly
486      Youssef Chermiti
487           Yunus Konak
Name: Player, Length: 488, dtype: object